<a href="https://colab.research.google.com/github/rakasiwisurya/pgaibllm-lessons/blob/main/Latihan_Membuat_Sistem_RAG_Sederhana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Persiapan Dependencies dan Ekosistem

In [ ]:
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.2.3
!pip install -q langchain-community==0.4.1
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:00


In [ ]:
import os
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Perangkat yang digunakan: {device}")

Perangkat yang digunakan: cpu


# Data Ingestion

## Loading

In [ ]:
!wget -O buku_panduan_gen_ai.pdf "https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb"

--2026-07-20 14:01:23--  https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.google.com (drive.google.com)... 172.217.204.113, 172.217.204.102, 172.217.204.100, ...
Connecting to drive.google.com (drive.google.com)|172.217.204.113|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb [following]
--2026-07-20 14:01:23--  https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 192.178.219.132, 2607:f8b0:400c:c0d::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|192.178.219.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 655208 (640K) [application/octet-stream]
Saving to: ‘buku_panduan_gen_ai.pdf’

buku_panduan_gen_ai 100%[===================>] 639.85K  --.-KB/s    in 0.02s   

2026-07-20 14:01:

In [ ]:
file_path = "/content/buku_panduan_gen_ai.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

In [ ]:
documents[8]

Document(metadata={'producer': 'iLovePDF', 'creator': 'Adobe InDesign 20.3 (Windows)', 'creationdate': '2025-06-03T13:36:46+07:00', 'source': '/content/buku_panduan_gen_ai.pdf', 'file_path': '/content/buku_panduan_gen_ai.pdf', 'total_pages': 42, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-06-04T05:01:35+00:00', 'trapped': '', 'modDate': 'D:20250604050135Z', 'creationDate': "D:20250603133646+07'00'", 'page': 8}, page_content='8\nBAB II: Pemahaman Teknologi Generative AI\n2.1. Perbedaan AI dan Generative AI\nSebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk \nmemahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-\ncara khusus.\n1.\t Artificial Intelligence (AI)\n•\t AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-\nagai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.\n•\t Contoh AI meliputi asisten virtual seperti Siri dan Goog

## Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "."]
)

# Proses Splitting
chunks = text_splitter.split_documents(documents)
print(f"Dokumen dipecah menjadi {len(chunks)} chunks.")

Dokumen dipecah menjadi 78 chunks.


## Storing

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': device}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

# Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
query = "Apa itu Generative AI"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
8
BAB II: Pemahaman Teknologi Generative AI
2.1. Perbedaan AI dan Generative AI
Sebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk 
memahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-
cara khusus.
1.	 Artificial Intelligence (AI)
•	 AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-
agai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.
•	 Contoh AI meliputi asisten virtual seperti Siri dan Google Assistant, algoritma pen-
carian Google, dan sistem rekomendasi Netflix atau Spotify.
•	 AI umumnya bersifat berbasis aturan atau pembelajaran mesin yang membantu ma-
nusia dalam mengambil keputusan.
2. Generative AI (GenAI)
•	 GenAI adalah cabang dari AI yang mampu menghasilkan konten baru, seperti teks, 
gambar, video, dan suara, berdasarkan data yang telah dipelajarinya.
•	 Model GenAI menggunakan teknik deep learning dan neural networks untuk men-

--- Dokumen 2 -

# Generation

## Panggil Model Generation (LLM)

In [ ]:
# Konfigurasi Kuantisasi 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model_name = "unsloth/Llama-3.2-3B-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

## Bangun Pipeline Generation

In [ ]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1000,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cpu


## Buat Template Prompt dan Chain

In [ ]:
# Llama 3 menggunakan format <|start_header_id|>system<|end_header_id|> dst.
template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Anda adalah asisten AI yang bertugas membantu pengguna.
Gunakan hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan.
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jujur bahwa Anda tidak mengetahui jawabannya dan jangan membuat asumsi atau jawaban tambahan.
Berikan jawaban secara singkat dan jelas.

Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>

{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "query"]
)

In [ ]:
rag_chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Testing

In [ ]:
def ask_question(query):
    print(f"Pertanyaan: {query}")

    # Jalankan Chain
    response = rag_chain.invoke(query)

    print("Jawaban:")
    print(response)

    # Tampilkan sumber referensi
    docs = retriever.invoke(query)
    print("\nSumber Referensi:")
    for i, doc in enumerate(docs):
        print(f"{i+1}. Halaman {doc.metadata.get('page', '?')}")

In [ ]:
query = "Apa itu Generative AI?"
ask_question(query)

Pertanyaan: Apa itu Generative AI?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Jawaban:
Generative AI (GenAI) adalah cabang dari AI yang dirancang untuk menghasilkan konten baru, seperti teks, gambar, suara, dan video, berdasarkan data yang telah dipelajarinya.

Sumber Referensi:
1. Halaman 8
2. Halaman 38
3. Halaman 9
